# Qwen2.5-VL-3B layer-tuned hidden-evidence verification

This notebook tests whether model-specific layer tuning improves the 3B hallucination-verification result.

## Experimental rule

The previous cross-model result used a frozen relative-depth rule:

- layer 27 of 37 hidden states
- top 3 hidden-evidence patches
- sufficiency lift:
  \[
  p_{\mathrm{yes}}(\text{top 3 only})
  -
  p_{\mathrm{yes}}(\text{random 3 only})
  \]
- larger values mean grounded

That frozen transfer result remains the valid cross-model confirmation result.

This notebook performs a separate **3B-tuned analysis**:

1. Reuse the exact same 3B matched examples and val/eval split.
2. Recompute hidden-state movement for every patch at every hidden layer.
3. Keep `k=3`, the sufficiency intervention, and score direction fixed.
4. Rank all layers using only the 3B validation split.
5. Select a robust layer using a three-layer-smoothed validation objective.
6. Select the classification threshold on validation.
7. Evaluate once on the untouched 3B evaluation split.
8. Compare tuned 3B with the original fixed layer-27 result and clean confidence.

The tuned result must be reported separately from the frozen transfer result.


In [ ]:
# Cell 1: install dependencies
# Run once on a fresh pod, restart the kernel, then continue with Cell 2.

import sys
import subprocess

packages = [
    "numpy==1.26.4",
    "scipy==1.11.4",
    "pandas==2.2.2",
    "pillow==10.4.0",
    "matplotlib==3.9.2",
    "scikit-learn==1.4.2",
    "transformers==4.50.3",
    "datasets==2.19.2",
    "huggingface_hub==0.34.4",
    "accelerate==0.34.2",
    "sentencepiece",
    "requests",
    "tqdm",
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "pip",
    "setuptools",
    "wheel",
])
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "--no-cache-dir",
    *packages,
])

print("Install complete. Restart the kernel, then run Cell 2.")


In [ ]:
# Cell 2: setup, prior split, dataset, model, and shared helpers

import gc
import json
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFilter
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoProcessor

try:
    from transformers import Qwen2_5_VLForConditionalGeneration as QwenVLClass
except Exception:
    from transformers import Qwen2VLForConditionalGeneration as QwenVLClass

# ============================================================
# Configuration
# ============================================================

WORKSPACE = Path("/workspace")
PROJECT_DIR = WORKSPACE / "pope_3b_layer_tuned_verification"
OUT_DIR = PROJECT_DIR / "outputs"
CACHE_DIR = PROJECT_DIR / "cache"
INPUT_DIR = PROJECT_DIR / "input"
CLEAN_HIDDEN_DIR = CACHE_DIR / "clean_hidden"
ALL_LAYER_DIR = CACHE_DIR / "all_layer_patch_distances"
EVIDENCE_DIR = CACHE_DIR / "evidence_only"

for directory in [
    PROJECT_DIR,
    OUT_DIR,
    CACHE_DIR,
    INPUT_DIR,
    CLEAN_HIDDEN_DIR,
    ALL_LAYER_DIR,
    EVIDENCE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
DATASET_ID = "lmms-lab/POPE"

GEN_BATCH_SIZE = 6
MAX_IMAGE_SIDE = 384
GRID_SIZE = 6
BLUR_RADIUS = 14
PRIMARY_K = 3
BOOTSTRAP_ROUNDS = 3000
FIXED_LAYER = 27

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("Project:", PROJECT_DIR)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# ============================================================
# Extract the prior 3B result package
# ============================================================

def valid_zips(patterns):
    found = []
    for pattern in patterns:
        found.extend(WORKSPACE.glob(pattern))
    return [
        path
        for path in found
        if path.is_file() and zipfile.is_zipfile(path)
    ]

prior_dir = INPUT_DIR / "prior_3b"
selected_path = prior_dir / "selected_crossmodel_confirmation.csv"

if not selected_path.exists():
    candidates = valid_zips([
        "pope_crossmodel_3b_sufficiency_confirmation_RESULTS.zip",
        "*crossmodel*3b*sufficiency*RESULTS*.zip",
        "*crossmodel*3b*.zip",
    ])
    if not candidates:
        raise FileNotFoundError(
            "Upload pope_crossmodel_3b_sufficiency_confirmation_RESULTS.zip "
            "to /workspace, then rerun Cell 2."
        )
    prior_zip = max(candidates, key=lambda path: path.stat().st_size)
    prior_dir.mkdir(parents=True, exist_ok=True)
    print("Extracting prior 3B results:", prior_zip)
    with zipfile.ZipFile(prior_zip, "r") as archive:
        archive.extractall(prior_dir)

required = [
    prior_dir / "selected_crossmodel_confirmation.csv",
    prior_dir / "crossmodel_analysis.csv",
    prior_dir / "crossmodel_manifest.json",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing prior result files: " + ", ".join(missing))

selected = pd.read_csv(prior_dir / "selected_crossmodel_confirmation.csv")
prior_analysis = pd.read_csv(prior_dir / "crossmodel_analysis.csv")
with open(prior_dir / "crossmodel_manifest.json") as file:
    prior_manifest = json.load(file)

selected["row_idx"] = pd.to_numeric(selected.row_idx, errors="raise").astype(int)
selected["selected_index"] = pd.to_numeric(
    selected.selected_index, errors="raise"
).astype(int)
selected["pair_id"] = pd.to_numeric(selected.pair_id, errors="raise").astype(int)
selected["is_grounded"] = (
    selected.clean_group.astype(str) == "grounded_yes"
).astype(int)

print("\nReused split:")
print(selected.groupby(["split", "pair_category", "clean_group"]).size())
print("Examples:", len(selected))
print("Prior fixed layer:", prior_manifest.get("heatmap_layer"))

# ============================================================
# Dataset
# ============================================================

dataset = load_dataset(DATASET_ID, split="test")

def get_image(row_idx):
    row = dataset[int(row_idx)]
    image = row.get("image")
    if isinstance(image, Image.Image):
        return image.convert("RGB")
    if hasattr(image, "convert"):
        return image.convert("RGB")
    raise ValueError(f"No image found for POPE row {row_idx}")

# ============================================================
# Model
# ============================================================

device_name = (
    torch.cuda.get_device_name(0).lower()
    if torch.cuda.is_available()
    else ""
)
use_float16 = any(
    token in device_name
    for token in ["a40", "rtx", "a5000", "a6000"]
)
dtype = (
    torch.float16
    if torch.cuda.is_available() and use_float16
    else (
        torch.bfloat16
        if torch.cuda.is_available()
        else torch.float32
    )
)

print("\nLoading:", MODEL_ID)
print("dtype:", dtype)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False,
)
if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
    processor.tokenizer.padding_side = "left"

tokenizer = processor.tokenizer

model = QwenVLClass.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
if not torch.cuda.is_available():
    model = model.to("cpu")
model.eval()

def model_device():
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cpu")

def resize_for_vlm(image, max_side=MAX_IMAGE_SIDE):
    image = image.convert("RGB")
    width, height = image.size
    largest = max(width, height)
    if largest <= max_side:
        return image
    scale = max_side / float(largest)
    return image.resize(
        (
            max(1, int(round(width * scale))),
            max(1, int(round(height * scale))),
        ),
        Image.Resampling.LANCZOS,
    )

def collect_token_ids(words):
    ids = []
    for word in words:
        variants = [
            word,
            " " + word,
            word.capitalize(),
            " " + word.capitalize(),
        ]
        for variant in variants:
            tokens = tokenizer.encode(variant, add_special_tokens=False)
            if len(tokens) == 1:
                ids.append(tokens[0])
    return sorted(set(ids))

YES_IDS = collect_token_ids(["yes"])
NO_IDS = collect_token_ids(["no"])

def answer_prompt(question):
    return (
        "Answer only yes or no based only on the visible image.\n"
        f"Question: {question}"
    )

def build_messages(images, prompts):
    messages = []
    for prompt in prompts:
        messages.append([
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt},
                ],
            }
        ])
    texts = [
        processor.apply_chat_template(
            message,
            tokenize=False,
            add_generation_prompt=True,
        )
        for message in messages
    ]
    safe_images = [resize_for_vlm(image) for image in images]
    return texts, safe_images

@torch.inference_mode()
def qwen_yesno_hidden_batch(images, prompts, return_hidden=False):
    texts, safe_images = build_messages(images, prompts)
    inputs = processor(
        text=texts,
        images=safe_images,
        return_tensors="pt",
        padding=True,
    )
    inputs = {
        key: value.to(model_device()) if hasattr(value, "to") else value
        for key, value in inputs.items()
    }

    generated = model.generate(
        **inputs,
        do_sample=False,
        num_beams=1,
        max_new_tokens=2,
        return_dict_in_generate=True,
        output_scores=True,
        output_hidden_states=return_hidden,
        use_cache=True,
    )

    input_length = inputs["input_ids"].shape[1]
    generated_ids = generated.sequences[:, input_length:]
    raw = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    raw = [text.strip() for text in raw]

    first_scores = generated.scores[0].float()
    probabilities = torch.softmax(first_scores, dim=-1)
    yes_probability = probabilities[:, YES_IDS].sum(dim=-1)
    no_probability = probabilities[:, NO_IDS].sum(dim=-1)
    p_yes = (
        yes_probability
        / (yes_probability + no_probability + 1e-12)
    ).detach().cpu().numpy().astype(float)
    prediction = (p_yes >= 0.5).astype(int)

    hidden = None
    if return_hidden:
        hidden_tuple = generated.hidden_states[0]
        hidden = torch.stack(
            [
                layer_hidden[:, -1, :].detach().float().cpu()
                for layer_hidden in hidden_tuple
            ],
            dim=1,
        ).numpy().astype(np.float16)

    del inputs, generated, generated_ids, first_scores, probabilities
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "raw": raw,
        "p_yes": p_yes,
        "pred": prediction,
        "hidden": hidden,
    }

# ============================================================
# Image interventions
# ============================================================

def patch_boxes(width, height, grid=GRID_SIZE):
    boxes = []
    for grid_y in range(grid):
        for grid_x in range(grid):
            x1 = int(round(grid_x * width / grid))
            y1 = int(round(grid_y * height / grid))
            x2 = int(round((grid_x + 1) * width / grid))
            y2 = int(round((grid_y + 1) * height / grid))
            boxes.append((x1, y1, x2, y2))
    return boxes

def blur_patch(image, box):
    image = image.convert("RGB")
    output = image.copy()
    blurred = image.filter(ImageFilter.GaussianBlur(radius=BLUR_RADIUS))
    x1, y1, x2, y2 = [int(value) for value in box]
    output.paste(blurred.crop((x1, y1, x2, y2)), (x1, y1))
    return output

def keep_only_indices(image, indices):
    image = image.convert("RGB")
    boxes = patch_boxes(image.width, image.height)
    background = image.filter(
        ImageFilter.GaussianBlur(radius=BLUR_RADIUS * 1.6)
    )
    for patch_index in indices:
        x1, y1, x2, y2 = boxes[int(patch_index)]
        background.paste(
            image.crop((x1, y1, x2, y2)),
            (x1, y1),
        )
    return background

selected_lookup = {
    int(row.selected_index): row
    for _, row in selected.iterrows()
}

print("\nCell 2 complete.")


In [ ]:
# Cell 3: recompute clean hidden states and every-layer patch movement
#
# This is the main model-inference cell.
# It checkpoints after every selected example, so it is safe to rerun.

# ============================================================
# Clean hidden states for the exact prior selected split
# ============================================================

clean_records_path = OUT_DIR / "clean_recompute.csv"
if clean_records_path.exists():
    clean_records = pd.read_csv(clean_records_path)
else:
    clean_records = pd.DataFrame()

completed_clean = {
    int(path.stem.split("_")[-1])
    for path in CLEAN_HIDDEN_DIR.glob("selected_*.npy")
}

pending_selected = selected[
    ~selected.selected_index.astype(int).isin(completed_clean)
].reset_index(drop=True)

new_clean_rows = []

for start in tqdm(
    range(0, len(pending_selected), GEN_BATCH_SIZE),
    desc="clean hidden states",
):
    batch = pending_selected.iloc[start:start + GEN_BATCH_SIZE]
    images = [get_image(int(row.row_idx)) for _, row in batch.iterrows()]
    prompts = [answer_prompt(row.question) for _, row in batch.iterrows()]
    output = qwen_yesno_hidden_batch(
        images,
        prompts,
        return_hidden=True,
    )

    for index, (_, row) in enumerate(batch.iterrows()):
        selected_index = int(row.selected_index)
        np.save(
            CLEAN_HIDDEN_DIR / f"selected_{selected_index}.npy",
            output["hidden"][index],
        )
        new_clean_rows.append({
            "selected_index": selected_index,
            "row_idx": int(row.row_idx),
            "clean_recomputed_raw": output["raw"][index],
            "clean_recomputed_p_yes": float(output["p_yes"][index]),
            "clean_recomputed_pred": int(output["pred"][index]),
        })

    if new_clean_rows:
        clean_records = pd.concat(
            [clean_records, pd.DataFrame(new_clean_rows)],
            ignore_index=True,
        ).drop_duplicates("selected_index", keep="last")
        clean_records.to_csv(clean_records_path, index=False)
        new_clean_rows = []

example_clean = np.load(
    CLEAN_HIDDEN_DIR / f"selected_{int(selected.iloc[0].selected_index)}.npy"
)
N_HIDDEN_STATES = int(example_clean.shape[0])
CANDIDATE_LAYERS = list(range(N_HIDDEN_STATES))

print("Hidden states found:", N_HIDDEN_STATES)
print("Candidate layers:", CANDIDATE_LAYERS)

# ============================================================
# Every patch at every hidden layer
# ============================================================

distance_manifest_rows = []

for _, meta in tqdm(
    selected.sort_values("selected_index").iterrows(),
    total=len(selected),
    desc="all-layer patch evidence",
):
    selected_index = int(meta.selected_index)
    distance_path = ALL_LAYER_DIR / f"selected_{selected_index}_distances.npy"
    p_yes_path = ALL_LAYER_DIR / f"selected_{selected_index}_perturb_p_yes.npy"

    if distance_path.exists() and p_yes_path.exists():
        distances = np.load(distance_path)
        perturb_p_yes = np.load(p_yes_path)
        if (
            distances.shape == (GRID_SIZE * GRID_SIZE, N_HIDDEN_STATES)
            and perturb_p_yes.shape == (GRID_SIZE * GRID_SIZE,)
        ):
            distance_manifest_rows.append({
                "selected_index": selected_index,
                "row_idx": int(meta.row_idx),
                "n_patches": int(distances.shape[0]),
                "n_hidden_states": int(distances.shape[1]),
                "status": "cached",
            })
            continue

    image = get_image(int(meta.row_idx))
    boxes = patch_boxes(image.width, image.height)
    clean_hidden = np.asarray(
        np.load(CLEAN_HIDDEN_DIR / f"selected_{selected_index}.npy"),
        dtype=np.float32,
    )

    distances = np.zeros(
        (len(boxes), N_HIDDEN_STATES),
        dtype=np.float32,
    )
    perturb_p_yes = np.zeros(len(boxes), dtype=np.float32)

    for start in range(0, len(boxes), GEN_BATCH_SIZE):
        patch_indices = list(
            range(start, min(start + GEN_BATCH_SIZE, len(boxes)))
        )
        images = [
            blur_patch(image, boxes[patch_index])
            for patch_index in patch_indices
        ]
        prompts = [answer_prompt(meta.question)] * len(images)

        output = qwen_yesno_hidden_batch(
            images,
            prompts,
            return_hidden=True,
        )

        for local_index, patch_index in enumerate(patch_indices):
            perturbed_hidden = np.asarray(
                output["hidden"][local_index],
                dtype=np.float32,
            )
            difference = perturbed_hidden - clean_hidden
            numerator = np.linalg.norm(difference, axis=1)
            denominator = np.linalg.norm(clean_hidden, axis=1) + 1e-8
            distances[patch_index] = numerator / denominator
            perturb_p_yes[patch_index] = float(
                output["p_yes"][local_index]
            )

    np.save(distance_path, distances)
    np.save(p_yes_path, perturb_p_yes)

    distance_manifest_rows.append({
        "selected_index": selected_index,
        "row_idx": int(meta.row_idx),
        "n_patches": int(distances.shape[0]),
        "n_hidden_states": int(distances.shape[1]),
        "status": "completed",
    })

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

distance_manifest = pd.DataFrame(distance_manifest_rows)
distance_manifest.to_csv(
    OUT_DIR / "all_layer_distance_manifest.csv",
    index=False,
)

with open(OUT_DIR / "all_layer_manifest.json", "w") as file:
    json.dump({
        "model_id": MODEL_ID,
        "n_examples": int(len(selected)),
        "n_hidden_states": int(N_HIDDEN_STATES),
        "candidate_layers": CANDIDATE_LAYERS,
        "grid_size": int(GRID_SIZE),
        "k_fixed": int(PRIMARY_K),
        "score": "top3 evidence-only p_yes minus random3 evidence-only p_yes",
        "direction": "larger means grounded",
        "selection_data": "validation split only",
    }, file, indent=2)

print("\nCell 3 complete.")
print(distance_manifest.status.value_counts())


In [ ]:
# Cell 4: tune the 3B layer on validation only
#
# k=3, intervention type, and score direction remain fixed.
# Only the hidden layer is selected.

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    accuracy_score,
)

validation = selected[
    selected.split.astype(str) == "val"
].copy()

def evidence_key(selected_index, indices):
    ordered = tuple(sorted(int(value) for value in indices))
    return (
        f"{int(selected_index)}|"
        + ",".join(str(value) for value in ordered)
    )

def matched_random_indices(selected_index, layer, top_indices, k=PRIMARY_K):
    top_set = {int(value) for value in top_indices}
    rng = np.random.default_rng(
        SEED
        + int(selected_index) * 1009
        + int(layer) * 10007
        + int(k) * 101
    )
    permutation = rng.permutation(GRID_SIZE * GRID_SIZE).astype(int).tolist()
    chosen = [
        index
        for index in permutation
        if index not in top_set
    ][:k]
    if len(chosen) != k:
        raise RuntimeError("Could not create matched random patch set.")
    return chosen

# ============================================================
# Build validation mappings and deduplicated evidence-only tasks
# ============================================================

mapping_rows = []
task_lookup = {}

for _, meta in validation.iterrows():
    selected_index = int(meta.selected_index)
    distances = np.load(
        ALL_LAYER_DIR / f"selected_{selected_index}_distances.npy"
    )

    for layer in CANDIDATE_LAYERS:
        ranking = np.argsort(
            -distances[:, int(layer)],
            kind="stable",
        )
        top_indices = ranking[:PRIMARY_K].astype(int).tolist()
        random_indices = matched_random_indices(
            selected_index,
            layer,
            top_indices,
            PRIMARY_K,
        )

        top_key = evidence_key(selected_index, top_indices)
        random_key = evidence_key(selected_index, random_indices)

        mapping_rows.append({
            "selected_index": selected_index,
            "row_idx": int(meta.row_idx),
            "pair_id": int(meta.pair_id),
            "pair_category": str(meta.pair_category),
            "clean_group": str(meta.clean_group),
            "is_grounded": int(meta.is_grounded),
            "layer": int(layer),
            "top_indices_json": json.dumps(top_indices),
            "random_indices_json": json.dumps(random_indices),
            "top_key": top_key,
            "random_key": random_key,
        })

        task_lookup[top_key] = {
            "evidence_key": top_key,
            "selected_index": selected_index,
            "row_idx": int(meta.row_idx),
            "question": str(meta.question),
            "patch_indices_json": json.dumps(top_indices),
        }
        task_lookup[random_key] = {
            "evidence_key": random_key,
            "selected_index": selected_index,
            "row_idx": int(meta.row_idx),
            "question": str(meta.question),
            "patch_indices_json": json.dumps(random_indices),
        }

val_mapping = pd.DataFrame(mapping_rows)
val_mapping.to_csv(OUT_DIR / "validation_layer_mapping.csv", index=False)

val_tasks = pd.DataFrame(task_lookup.values()).sort_values(
    ["selected_index", "evidence_key"]
).reset_index(drop=True)
val_tasks.to_csv(OUT_DIR / "validation_evidence_tasks.csv", index=False)

# ============================================================
# Run deduplicated validation evidence-only images
# ============================================================

val_results_path = OUT_DIR / "validation_evidence_results.csv"
if val_results_path.exists():
    val_results = pd.read_csv(val_results_path)
else:
    val_results = pd.DataFrame()

completed = set(
    val_results.get(
        "evidence_key",
        pd.Series(dtype=str),
    ).astype(str).tolist()
)
pending = val_tasks[
    ~val_tasks.evidence_key.astype(str).isin(completed)
].reset_index(drop=True)

print("Validation layer candidates:", len(CANDIDATE_LAYERS))
print("Raw candidate interventions:", len(val_mapping) * 2)
print("Unique evidence-only images:", len(val_tasks))
print("Pending evidence-only images:", len(pending))

new_rows = []

for start in tqdm(
    range(0, len(pending), GEN_BATCH_SIZE),
    desc="validation sufficiency sweep",
):
    batch = pending.iloc[start:start + GEN_BATCH_SIZE]
    images = []
    prompts = []

    for _, task in batch.iterrows():
        original = get_image(int(task.row_idx))
        indices = [
            int(value)
            for value in json.loads(task.patch_indices_json)
        ]
        evidence_image = keep_only_indices(original, indices)
        images.append(evidence_image)
        prompts.append(answer_prompt(task.question))

    output = qwen_yesno_hidden_batch(
        images,
        prompts,
        return_hidden=False,
    )

    for index, (_, task) in enumerate(batch.iterrows()):
        new_rows.append({
            **task.to_dict(),
            "evidence_raw": output["raw"][index],
            "evidence_p_yes": float(output["p_yes"][index]),
            "evidence_pred_yes": int(output["pred"][index]),
        })

    if (
        len(new_rows) >= 120
        or start + GEN_BATCH_SIZE >= len(pending)
    ):
        val_results = pd.concat(
            [val_results, pd.DataFrame(new_rows)],
            ignore_index=True,
        ).drop_duplicates("evidence_key", keep="last")
        val_results.to_csv(val_results_path, index=False)
        new_rows = []

result_lookup = val_results.set_index("evidence_key")["evidence_p_yes"]

val_scored = val_mapping.copy()
val_scored["top3_p_yes"] = val_scored.top_key.map(result_lookup)
val_scored["random3_p_yes"] = val_scored.random_key.map(result_lookup)
val_scored["verification_score"] = (
    val_scored.top3_p_yes
    - val_scored.random3_p_yes
)

if val_scored[["top3_p_yes", "random3_p_yes"]].isna().any().any():
    raise RuntimeError("Some validation intervention results are missing.")

val_scored.to_csv(
    OUT_DIR / "validation_layer_scores_long.csv",
    index=False,
)

# ============================================================
# Layer metrics
# ============================================================

def safe_auc(y, scores):
    y = np.asarray(y, dtype=int)
    scores = np.asarray(scores, dtype=float)
    if len(np.unique(y)) < 2:
        return float("nan")
    return float(roc_auc_score(y, scores))

layer_rows = []

for layer, frame in val_scored.groupby("layer", sort=True):
    y = frame.is_grounded.to_numpy(int)
    scores = frame.verification_score.to_numpy(float)

    category_aucs = []
    for category, category_frame in frame.groupby("pair_category"):
        category_aucs.append(
            safe_auc(
                category_frame.is_grounded,
                category_frame.verification_score,
            )
        )

    pair_table = frame.pivot(
        index="pair_id",
        columns="clean_group",
        values="verification_score",
    )
    pair_table["pair_gap"] = (
        pair_table["grounded_yes"]
        - pair_table["hallucinated_yes"]
    )

    layer_rows.append({
        "layer": int(layer),
        "depth_fraction": float(layer / max(1, N_HIDDEN_STATES - 1)),
        "pooled_auroc": safe_auc(y, scores),
        "pooled_auprc": float(average_precision_score(y, scores)),
        "mean_category_auroc": float(np.nanmean(category_aucs)),
        "min_category_auroc": float(np.nanmin(category_aucs)),
        "paired_win_rate": float((pair_table.pair_gap > 0).mean()),
        "mean_pair_gap": float(pair_table.pair_gap.mean()),
        "median_pair_gap": float(pair_table.pair_gap.median()),
    })

layer_metrics = pd.DataFrame(layer_rows).sort_values("layer").reset_index(drop=True)

# Smooth only the selection objective to avoid choosing a one-layer spike.
layer_metrics["smoothed_mean_category_auroc"] = (
    layer_metrics.mean_category_auroc
    .rolling(window=3, center=True, min_periods=2)
    .mean()
)
layer_metrics["smoothed_pooled_auroc"] = (
    layer_metrics.pooled_auroc
    .rolling(window=3, center=True, min_periods=2)
    .mean()
)

layer_metrics["distance_from_fixed_layer"] = (
    layer_metrics.layer.astype(int) - FIXED_LAYER
).abs()

ranked_layers = layer_metrics.sort_values(
    [
        "smoothed_mean_category_auroc",
        "smoothed_pooled_auroc",
        "paired_win_rate",
        "mean_pair_gap",
        "distance_from_fixed_layer",
    ],
    ascending=[False, False, False, False, True],
).reset_index(drop=True)

SELECTED_LAYER = int(ranked_layers.iloc[0].layer)

layer_metrics["selected"] = (
    layer_metrics.layer.astype(int) == SELECTED_LAYER
).astype(int)
layer_metrics.to_csv(
    OUT_DIR / "validation_layer_sweep_metrics.csv",
    index=False,
)

print("\nTop validation layers")
print(
    ranked_layers[
        [
            "layer",
            "depth_fraction",
            "mean_category_auroc",
            "smoothed_mean_category_auroc",
            "pooled_auroc",
            "paired_win_rate",
            "mean_pair_gap",
        ]
    ].head(10).to_string(index=False)
)
print("\nSelected tuned 3B layer:", SELECTED_LAYER)
print(
    "Depth fraction:",
    SELECTED_LAYER / max(1, N_HIDDEN_STATES - 1),
)

# ============================================================
# Threshold selection on validation
# ============================================================

def choose_threshold(y_true, scores):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    unique_scores = np.unique(scores)

    if len(unique_scores) == 1:
        candidates = [unique_scores[0]]
    else:
        candidates = np.concatenate([
            [unique_scores[0] - 1e-9],
            (unique_scores[:-1] + unique_scores[1:]) / 2.0,
            [unique_scores[-1] + 1e-9],
        ])

    best = None
    for threshold in candidates:
        predictions = (scores >= threshold).astype(int)
        candidate = (
            float(balanced_accuracy_score(y_true, predictions)),
            float(accuracy_score(y_true, predictions)),
            float(threshold),
        )
        if best is None or candidate[:2] > best[:2]:
            best = candidate

    return {
        "threshold": best[2],
        "balanced_accuracy": best[0],
        "accuracy": best[1],
    }

selected_val = val_scored[
    val_scored.layer.astype(int) == SELECTED_LAYER
].copy()

threshold_result = choose_threshold(
    selected_val.is_grounded,
    selected_val.verification_score,
)
TUNED_THRESHOLD = float(threshold_result["threshold"])

tuned_config = {
    "model_id": MODEL_ID,
    "n_hidden_states": int(N_HIDDEN_STATES),
    "fixed_reference_layer": int(FIXED_LAYER),
    "selected_layer": int(SELECTED_LAYER),
    "selected_depth_fraction": float(
        SELECTED_LAYER / max(1, N_HIDDEN_STATES - 1)
    ),
    "k": int(PRIMARY_K),
    "score": "top3_p_yes minus random3_p_yes",
    "direction": "larger means grounded",
    "selection_objective": (
        "three-layer-smoothed mean validation category AUROC, "
        "then smoothed pooled AUROC, paired win rate, pair gap"
    ),
    "validation_threshold": float(TUNED_THRESHOLD),
    "validation_threshold_balanced_accuracy": float(
        threshold_result["balanced_accuracy"]
    ),
}

with open(OUT_DIR / "tuned_3b_config.json", "w") as file:
    json.dump(tuned_config, file, indent=2)

# Layer sweep figure
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    layer_metrics.layer,
    layer_metrics.mean_category_auroc,
    marker="o",
    markersize=3,
    label="Mean category AUROC",
)
ax.plot(
    layer_metrics.layer,
    layer_metrics.smoothed_mean_category_auroc,
    linewidth=2,
    label="Three-layer smoothed",
)
ax.axvline(
    FIXED_LAYER,
    linestyle="--",
    label=f"Fixed layer {FIXED_LAYER}",
)
ax.axvline(
    SELECTED_LAYER,
    linestyle=":",
    label=f"Tuned layer {SELECTED_LAYER}",
)
ax.axhline(0.5, linestyle="-.", label="Chance")
ax.set_xlabel("Hidden-state index")
ax.set_ylabel("Validation AUROC")
ax.set_title("Qwen2.5-VL-3B layer tuning on validation only")
ax.legend()
fig.tight_layout()
fig.savefig(
    OUT_DIR / "validation_layer_sweep.png",
    dpi=220,
    bbox_inches="tight",
)
plt.close(fig)

print("\nCell 4 complete. Run Cell 5 for the untouched evaluation split.")


In [ ]:
# Cell 5: one held-out evaluation at the tuned layer, comparison, and export

from scipy.stats import wilcoxon
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    accuracy_score,
)
import matplotlib.pyplot as plt

evaluation = selected[
    selected.split.astype(str) == "eval"
].copy()

# ============================================================
# Build tuned-layer evaluation tasks
# ============================================================

eval_mapping_rows = []
eval_task_lookup = {}

for _, meta in evaluation.iterrows():
    selected_index = int(meta.selected_index)
    distances = np.load(
        ALL_LAYER_DIR / f"selected_{selected_index}_distances.npy"
    )
    ranking = np.argsort(
        -distances[:, SELECTED_LAYER],
        kind="stable",
    )
    top_indices = ranking[:PRIMARY_K].astype(int).tolist()
    random_indices = matched_random_indices(
        selected_index,
        SELECTED_LAYER,
        top_indices,
        PRIMARY_K,
    )

    top_key = evidence_key(selected_index, top_indices)
    random_key = evidence_key(selected_index, random_indices)

    eval_mapping_rows.append({
        "selected_index": selected_index,
        "row_idx": int(meta.row_idx),
        "pair_id": int(meta.pair_id),
        "pair_category": str(meta.pair_category),
        "clean_group": str(meta.clean_group),
        "is_grounded": int(meta.is_grounded),
        "clean_p_yes": float(meta.clean_p_yes),
        "layer": int(SELECTED_LAYER),
        "top_indices_json": json.dumps(top_indices),
        "random_indices_json": json.dumps(random_indices),
        "top_key": top_key,
        "random_key": random_key,
    })

    eval_task_lookup[top_key] = {
        "evidence_key": top_key,
        "selected_index": selected_index,
        "row_idx": int(meta.row_idx),
        "question": str(meta.question),
        "patch_indices_json": json.dumps(top_indices),
    }
    eval_task_lookup[random_key] = {
        "evidence_key": random_key,
        "selected_index": selected_index,
        "row_idx": int(meta.row_idx),
        "question": str(meta.question),
        "patch_indices_json": json.dumps(random_indices),
    }

eval_mapping = pd.DataFrame(eval_mapping_rows)
eval_mapping.to_csv(OUT_DIR / "evaluation_tuned_mapping.csv", index=False)

eval_tasks = pd.DataFrame(eval_task_lookup.values()).sort_values(
    ["selected_index", "evidence_key"]
).reset_index(drop=True)
eval_tasks.to_csv(OUT_DIR / "evaluation_tuned_tasks.csv", index=False)

# ============================================================
# Run exactly the selected configuration on evaluation
# ============================================================

eval_results_path = OUT_DIR / "evaluation_tuned_evidence_results.csv"
if eval_results_path.exists():
    eval_results = pd.read_csv(eval_results_path)
else:
    eval_results = pd.DataFrame()

completed = set(
    eval_results.get(
        "evidence_key",
        pd.Series(dtype=str),
    ).astype(str).tolist()
)
pending = eval_tasks[
    ~eval_tasks.evidence_key.astype(str).isin(completed)
].reset_index(drop=True)

print("Tuned evaluation layer:", SELECTED_LAYER)
print("Evaluation evidence-only images:", len(eval_tasks))
print("Pending:", len(pending))

new_rows = []

for start in tqdm(
    range(0, len(pending), GEN_BATCH_SIZE),
    desc="tuned 3B held-out evaluation",
):
    batch = pending.iloc[start:start + GEN_BATCH_SIZE]
    images = []
    prompts = []

    for _, task in batch.iterrows():
        original = get_image(int(task.row_idx))
        indices = [
            int(value)
            for value in json.loads(task.patch_indices_json)
        ]
        images.append(keep_only_indices(original, indices))
        prompts.append(answer_prompt(task.question))

    output = qwen_yesno_hidden_batch(
        images,
        prompts,
        return_hidden=False,
    )

    for index, (_, task) in enumerate(batch.iterrows()):
        new_rows.append({
            **task.to_dict(),
            "evidence_raw": output["raw"][index],
            "evidence_p_yes": float(output["p_yes"][index]),
            "evidence_pred_yes": int(output["pred"][index]),
        })

    if (
        len(new_rows) >= 120
        or start + GEN_BATCH_SIZE >= len(pending)
    ):
        eval_results = pd.concat(
            [eval_results, pd.DataFrame(new_rows)],
            ignore_index=True,
        ).drop_duplicates("evidence_key", keep="last")
        eval_results.to_csv(eval_results_path, index=False)
        new_rows = []

lookup = eval_results.set_index("evidence_key")["evidence_p_yes"]

tuned_eval = eval_mapping.copy()
tuned_eval["top3_p_yes"] = tuned_eval.top_key.map(lookup)
tuned_eval["random3_p_yes"] = tuned_eval.random_key.map(lookup)
tuned_eval["verification_score"] = (
    tuned_eval.top3_p_yes - tuned_eval.random3_p_yes
)

if tuned_eval[["top3_p_yes", "random3_p_yes"]].isna().any().any():
    raise RuntimeError("Some tuned evaluation intervention results are missing.")

tuned_eval.to_csv(
    OUT_DIR / "tuned_3b_evaluation_analysis.csv",
    index=False,
)

# ============================================================
# Metrics
# ============================================================

def metric_dict(frame, score_column, threshold):
    y = frame.is_grounded.to_numpy(int)
    scores = frame[score_column].to_numpy(float)
    predictions = (scores >= float(threshold)).astype(int)
    return {
        "n": int(len(frame)),
        "auroc": float(roc_auc_score(y, scores)),
        "auprc": float(average_precision_score(y, scores)),
        "balanced_accuracy": float(
            balanced_accuracy_score(y, predictions)
        ),
        "accuracy": float(accuracy_score(y, predictions)),
        "grounded_accept_rate": float(predictions[y == 1].mean()),
        "hallucination_reject_rate": float(
            1.0 - predictions[y == 0].mean()
        ),
    }

tuned_metrics = metric_dict(
    tuned_eval,
    "verification_score",
    TUNED_THRESHOLD,
)

# Category metrics
category_rows = []
for category, frame in tuned_eval.groupby("pair_category"):
    result = metric_dict(
        frame,
        "verification_score",
        TUNED_THRESHOLD,
    )
    result["pair_category"] = str(category)
    category_rows.append(result)

category_metrics = pd.DataFrame(category_rows)
category_metrics.to_csv(
    OUT_DIR / "tuned_3b_category_metrics.csv",
    index=False,
)

# Pairwise test
pair_table = tuned_eval.pivot(
    index="pair_id",
    columns="clean_group",
    values="verification_score",
)
pair_table["pair_gap"] = (
    pair_table["grounded_yes"]
    - pair_table["hallucinated_yes"]
)

try:
    paired_test = wilcoxon(
        pair_table.pair_gap.to_numpy(float),
        alternative="greater",
        zero_method="wilcox",
    )
    paired_p = float(paired_test.pvalue)
except Exception:
    paired_p = float("nan")

# Pair bootstrap so both examples from a matched pair stay together.
pair_ids = tuned_eval.pair_id.drop_duplicates().astype(int).to_numpy()
rng = np.random.default_rng(SEED + 99991)
bootstrap_aurocs = []
bootstrap_balanced = []

for _ in range(BOOTSTRAP_ROUNDS):
    sampled_ids = rng.choice(pair_ids, size=len(pair_ids), replace=True)
    parts = []
    for bootstrap_pair_index, pair_id in enumerate(sampled_ids):
        part = tuned_eval[
            tuned_eval.pair_id.astype(int) == int(pair_id)
        ].copy()
        part["bootstrap_pair_index"] = bootstrap_pair_index
        parts.append(part)
    sample = pd.concat(parts, ignore_index=True)

    y = sample.is_grounded.to_numpy(int)
    scores = sample.verification_score.to_numpy(float)
    predictions = (scores >= TUNED_THRESHOLD).astype(int)

    bootstrap_aurocs.append(float(roc_auc_score(y, scores)))
    bootstrap_balanced.append(
        float(balanced_accuracy_score(y, predictions))
    )

tuned_metrics["auroc_ci_95"] = [
    float(np.quantile(bootstrap_aurocs, 0.025)),
    float(np.quantile(bootstrap_aurocs, 0.975)),
]
tuned_metrics["balanced_accuracy_ci_95"] = [
    float(np.quantile(bootstrap_balanced, 0.025)),
    float(np.quantile(bootstrap_balanced, 0.975)),
]
tuned_metrics["paired_grounded_win_rate"] = float(
    (pair_table.pair_gap > 0).mean()
)
tuned_metrics["paired_mean_gap"] = float(pair_table.pair_gap.mean())
tuned_metrics["paired_wilcoxon_p"] = paired_p

# ============================================================
# Original fixed-layer result and clean-confidence baseline
# ============================================================

prior_analysis["is_grounded"] = (
    prior_analysis.clean_group.astype(str) == "grounded_yes"
).astype(int)

prior_val = prior_analysis[
    prior_analysis.split.astype(str) == "val"
].copy()
prior_eval = prior_analysis[
    prior_analysis.split.astype(str) == "eval"
].copy()

fixed_threshold_result = choose_threshold(
    prior_val.is_grounded,
    prior_val.frozen_score,
)
fixed_metrics = metric_dict(
    prior_eval.rename(columns={"frozen_score": "score"}),
    "score",
    fixed_threshold_result["threshold"],
)

confidence_threshold_result = choose_threshold(
    prior_val.is_grounded,
    prior_val.clean_p_yes,
)
confidence_metrics = metric_dict(
    prior_eval.rename(columns={"clean_p_yes": "score"}),
    "score",
    confidence_threshold_result["threshold"],
)

comparison = pd.DataFrame([
    {
        "method": f"Fixed layer {FIXED_LAYER}",
        **fixed_metrics,
    },
    {
        "method": f"Tuned layer {SELECTED_LAYER}",
        **{
            key: value
            for key, value in tuned_metrics.items()
            if not isinstance(value, list)
        },
    },
    {
        "method": "Clean confidence",
        **confidence_metrics,
    },
])
comparison.to_csv(
    OUT_DIR / "heldout_method_comparison.csv",
    index=False,
)

# Held-out comparison figure
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    comparison.method,
    comparison.auroc,
)
ax.axhline(0.5, linestyle="--")
ax.set_ylim(0.0, 1.0)
ax.set_ylabel("Held-out AUROC")
ax.set_title("Qwen2.5-VL-3B verification")
ax.tick_params(axis="x", rotation=18)
fig.tight_layout()
fig.savefig(
    OUT_DIR / "heldout_3b_comparison.png",
    dpi=220,
    bbox_inches="tight",
)
plt.close(fig)

final_result = {
    "status": "complete",
    "interpretation": (
        "Model-specific 3B tuning. This result does not replace the "
        "original frozen cross-model transfer result."
    ),
    "fixed_reference": {
        "layer": int(FIXED_LAYER),
        "metrics": fixed_metrics,
    },
    "tuned_3b": {
        "selected_layer": int(SELECTED_LAYER),
        "selected_depth_fraction": float(
            SELECTED_LAYER / max(1, N_HIDDEN_STATES - 1)
        ),
        "k": int(PRIMARY_K),
        "validation_threshold": float(TUNED_THRESHOLD),
        "metrics": tuned_metrics,
        "category_metrics": category_metrics.to_dict(orient="records"),
    },
    "clean_confidence": confidence_metrics,
}

with open(OUT_DIR / "tuned_3b_final_result.json", "w") as file:
    json.dump(final_result, file, indent=2)

macros = rf"""
\newcommand{{\TunedThreeBLayer}}{{{SELECTED_LAYER}}}
\newcommand{{\TunedThreeBDepth}}{{{SELECTED_LAYER / max(1, N_HIDDEN_STATES - 1):.3f}}}
\newcommand{{\TunedThreeBAUROC}}{{{tuned_metrics['auroc']:.3f}}}
\newcommand{{\TunedThreeBAUPRC}}{{{tuned_metrics['auprc']:.3f}}}
\newcommand{{\TunedThreeBBalAcc}}{{{tuned_metrics['balanced_accuracy']:.3f}}}
\newcommand{{\FixedThreeBAUROC}}{{{fixed_metrics['auroc']:.3f}}}
"""
(OUT_DIR / "tuned_3b_macros.tex").write_text(macros.strip() + "\n")

# ============================================================
# Export compact result package
# ============================================================

package_dir = PROJECT_DIR / "pope_3b_layer_tuned_verification_RESULTS"
if package_dir.exists():
    shutil.rmtree(package_dir)
package_dir.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    "clean_recompute.csv",
    "all_layer_distance_manifest.csv",
    "all_layer_manifest.json",
    "validation_layer_mapping.csv",
    "validation_evidence_tasks.csv",
    "validation_evidence_results.csv",
    "validation_layer_scores_long.csv",
    "validation_layer_sweep_metrics.csv",
    "validation_layer_sweep.png",
    "tuned_3b_config.json",
    "evaluation_tuned_mapping.csv",
    "evaluation_tuned_tasks.csv",
    "evaluation_tuned_evidence_results.csv",
    "tuned_3b_evaluation_analysis.csv",
    "tuned_3b_category_metrics.csv",
    "heldout_method_comparison.csv",
    "heldout_3b_comparison.png",
    "tuned_3b_final_result.json",
    "tuned_3b_macros.tex",
]

for name in files_to_copy:
    source = OUT_DIR / name
    if source.exists():
        shutil.copy2(source, package_dir / name)

archive_base = WORKSPACE / "pope_3b_layer_tuned_verification_RESULTS"
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=package_dir,
)

print("\nFINAL TUNED 3B RESULT")
print(json.dumps(final_result, indent=2))
print("\nResult zip:", archive_path)
